# Final model: horizon features, regularised trees, bagged over 5 seeds

The submission model. Feature set `horizon_nolevel` (notebook 09: scale-free margin and reservoir features, absolute levels
dropped, context windows and expanding-window horizon statistics of the scale-free margin; 384 columns). Tree parameters from
the parameter sweep under rolling validation (`min_child_weight` 50, `reg_lambda` 20, `colsample_bytree` 0.5). The number of
rounds is chosen by early stopping on 2021, the 2022 report score comes from one model trained on cases before 2022, and the
final prediction is the average of five models trained on all cases with different `random_state` (bagging over the row and
column subsampling, Predictor-practice slides).

### Shared code

Data loading, feature construction and the honest validation protocol. The same code is repeated at the top of every notebook so each one runs on its own.

In [ ]:
from pathlib import Path
from scipy.ndimage import uniform_filter1d, maximum_filter1d, minimum_filter1d
from sklearn.metrics import roc_auc_score
import json
import numpy as np
import pandas as pd
import re
import time
import xgboost as xgb

# ---- formerly src/train_xgb_baseline.py ----
# XGBoost baseline for the SINTEF unit-commitment task.
#
# Script version of xgboost_baseline.ipynb so it can run headless on the GPU desktop.
#
# Validation protocol: rounds are chosen by early stopping on 2021 (training on cases before 2021);
# the reported score is then from a model trained on cases before 2022 and scored on 2022, so the
# reported year never influences a modelling decision. A one-week gap keeps overlapping cases apart.
#
#     py train_xgb_baseline.py --device cuda          # GPU
#     python train_xgb_baseline.py --device cpu       # laptop
#
# Outputs (in --out, default ./results):
#     submission_xgb_baseline.csv   Kaggle submission (test cases 2023-2024)
#     val_pred_2022.npy             hold-out probabilities, shape (365*168, 14)
#     metrics.json                  hold-out micro AUC, per-unit AUC, best iteration, timings
#     feature_importance.csv        gain per feature (summed over the 14 boosters)

from kaggle_metric import score

H = 168
SEED = 42

def load(data_dir):
    data_dir = Path(data_dir)
    uc = pd.read_csv(
        data_dir / "Unit_commitment_decisions.csv", parse_dates=["starttime"]
    )
    price = (
        pd.read_csv(
            data_dir / "Historical_day_ahead_price_2015_2025.csv", parse_dates=["date"]
        )
        .set_index("date")["Day-ahead price (EUR/MWh)"]
        .resample("h")
        .mean()
    )
    inflow = pd.read_csv(
        data_dir / "Historical_inflow_1958_2025.csv", parse_dates=["date"]
    ).set_index("date")
    inflow = inflow[inflow.index >= "2014-12-01"].resample("h").mean()
    volume = pd.read_csv(
        data_dir / "Historical_volume_2015_2024.csv", parse_dates=["date"]
    ).set_index("date")
    water_value = pd.read_csv(
        data_dir / "Synthetic_water_value_2015_2024.csv", parse_dates=["date"]
    ).set_index("date")
    return uc, price, inflow, volume, water_value

class FeatureBuilder:
    def __init__(self, price, inflow, volume, water_value):
        self.price, self.inflow, self.volume, self.wv = (
            price,
            inflow,
            volume,
            water_value,
        )

    @staticmethod
    def window(hourly_df, starts, horizon=H):
        idx = hourly_df.index.get_indexer(starts)
        assert (idx >= 0).all(), (
            "some case start times are missing from the hourly series"
        )
        pos = idx[:, None] + np.arange(horizon)[None, :]
        return hourly_df.to_numpy(dtype=np.float32)[pos]

    def __call__(self, starts):
        starts = pd.DatetimeIndex(starts)
        if starts.tz is None:
            starts = starts.tz_localize("UTC")
        n = len(starts)
        P = self.window(self.price.to_frame(), starts)[:, :, 0]
        I = self.window(self.inflow, starts)
        V = self.volume.reindex(starts).to_numpy(dtype=np.float32)
        W = self.wv.reindex(starts).to_numpy(dtype=np.float32)
        W_end = self.wv.reindex(starts + pd.Timedelta(days=7)).to_numpy(
            dtype=np.float32
        )
        hours = pd.DatetimeIndex(
            (
                starts.values[:, None] + np.arange(H)[None, :] * np.timedelta64(1, "h")
            ).ravel()
        )
        rep = lambda a: np.repeat(a, H)

        f = pd.DataFrame(
            {
                "t": np.tile(np.arange(H), n),
                "hour": hours.hour,
                "dow": hours.dayofweek,
                "month": hours.month,
                "doy": hours.dayofyear,
                "price": P.ravel(),
                "price_week_mean": rep(P.mean(1)),
                "price_week_std": rep(P.std(1)),
                "price_week_min": rep(P.min(1)),
                "price_week_max": rep(P.max(1)),
            }
        )
        f["price_rel"] = f["price"] - f["price_week_mean"]
        f["price_rank"] = (P.argsort(1).argsort(1) / (H - 1)).ravel()
        f["price_day_mean"] = np.repeat(P.reshape(n, 7, 24).mean(2), 24, axis=1).ravel()
        f["price_prev"] = np.concatenate([P[:, :1], P[:, :-1]], axis=1).ravel()   # edge-padded, no wrap
        f["price_next"] = np.concatenate([P[:, 1:], P[:, -1:]], axis=1).ravel()
        for j, c in enumerate(self.inflow.columns):
            f[f"inflow_{c}"] = I[:, :, j].ravel()
        f["inflow_total"] = I.sum(2).ravel()
        for j, c in enumerate(self.volume.columns):
            f[f"vol_{c}"] = rep(V[:, j])
        for j, c in enumerate(self.wv.columns):
            f[f"wv_{c}"] = rep(W[:, j])
        for j, c in enumerate(self.wv.columns):
            f[f"wv_end_{c}"] = rep(W_end[:, j])
        return f

def split_masks(starts, val_year, gap_days=7):
    """Case-level masks. train: cases whose whole week ends before val_year (so no hour is shared with the
    validation cases); val: cases starting in val_year. Returned per case, use np.repeat(mask, H) for rows."""
    starts = pd.DatetimeIndex(starts)
    val_start = pd.Timestamp(f"{val_year}-01-01")
    train = np.asarray(starts < val_start - pd.Timedelta(days=gap_days - 1))
    val = np.asarray(starts.year == val_year)
    return train, val

def honest_fit(X, Y, starts, params, tune_year=2021, report_year=2022, device="cuda", verbose=False, max_rounds=3000):
    """Two-stage validation so the reported year is never used to make a modelling decision.

    stage 1: train on cases before tune_year, early-stop on tune_year  -> number of rounds
    stage 2: train on cases before report_year with that many rounds   -> score on report_year
    Returns a dict with n_rounds, tune_auc, report_auc, per-unit AUC, the stage-2 model and its predictions.
    """
    common = dict(tree_method="hist", device=device, random_state=SEED, **params)
    tr1, va1 = (np.repeat(m, H) for m in split_masks(starts, tune_year))
    m1 = xgb.XGBClassifier(n_estimators=max_rounds, eval_metric="auc", early_stopping_rounds=100, **common)
    m1.fit(X[tr1], Y[tr1], eval_set=[(X[va1], Y[va1])], verbose=verbose)
    n_rounds = int(m1.best_iteration) + 1
    tune_auc = roc_auc_score(Y[va1], m1.predict_proba(X[va1]), average="micro")

    tr2, va2 = (np.repeat(m, H) for m in split_masks(starts, report_year))
    m2 = xgb.XGBClassifier(n_estimators=n_rounds, **common).fit(X[tr2], Y[tr2])
    pred = m2.predict_proba(X[va2])
    report_auc = roc_auc_score(Y[va2], pred, average="micro")   # identical to kaggle_metric.score
    per_unit = [roc_auc_score(Y[va2][:, j], pred[:, j]) for j in range(Y.shape[1])]
    return dict(n_rounds=n_rounds, tune_auc=float(tune_auc), report_auc=float(report_auc),
                per_unit=per_unit, model=m2, pred=pred, val_mask=va2)

# ---- formerly src/features_margin.py ----
# Price-minus-water-value features (see xgboost_price_minus_wv.ipynb).
# plant -> reservoir directly upstream, from the connections in data/extended/Tokke_Vinje_topology.yaml
UPSTREAM = {
    "Hogga": "Bandak", "Tokke": "Vinjevatn", "Lio": "Byrtevatn", "Byrte": "Botnedalsvatn",
    "Vinje": "Vaamarvatn", "Haukeli": "Vatjern", "Songa": "Songavatn", "Kjela": "Foersvatn",
    "Vesle Kjela": "Kjelavatn",
}

def add_margin_features(f):
    f = f.copy()
    for plant, res in UPSTREAM.items():
        f[f"margin_{plant}"] = f["price"] - f[f"wv_{res}"]
        f[f"margin_end_{plant}"] = f["price"] - f[f"wv_end_{res}"]
        f[f"margin_end_wk_{plant}"] = f["price_week_mean"] - f[f"wv_end_{res}"]
    return f

# ---- formerly src/features_constraints.py ----
# Environmental-constraint features from data/extended (see xgboost_constraints.ipynb).
#
# Hourly minimum river flows (6 rivers) and minimum reservoir volumes (3 reservoirs), plus the distance between the
# reservoir's volume at case start and its minimum. All are inputs the optimiser receives for the case week.

H = 168

def load_constraints(data_dir="data/extended"):
    data_dir = Path(data_dir)
    minflow = pd.read_csv(data_dir / "Constraint_min_flow.csv", parse_dates=["date"]).set_index("date").resample("h").mean()
    minvol = pd.read_csv(data_dir / "Constraint_min_volume.csv", parse_dates=["date"]).set_index("date").resample("h").mean()
    return minflow, minvol

class ConstraintFeatures:
    def __init__(self, minflow, minvol, volume):
        self.minflow, self.minvol, self.volume = minflow, minvol, volume

    @staticmethod
    def window(hourly_df, starts):
        idx = hourly_df.index.get_indexer(starts)
        assert (idx >= 0).all(), "case start missing from constraint series"
        return hourly_df.to_numpy(dtype=np.float32)[idx[:, None] + np.arange(H)[None, :]]   # (n, H, cols)

    def __call__(self, starts):
        starts = pd.DatetimeIndex(starts)
        if starts.tz is None:
            starts = starts.tz_localize("UTC")
        n = len(starts)
        rep = lambda a: np.repeat(a, H)
        MF = self.window(self.minflow, starts)
        MV = self.window(self.minvol, starts)
        V = self.volume.reindex(starts)

        f = {}
        for j, river in enumerate(self.minflow.columns):
            f[f"minflow_{river}"] = MF[:, :, j].ravel()                 # requirement at hour t
            f[f"minflow_{river}_wkmax"] = rep(MF[:, :, j].max(1))       # tightest requirement during the week
        for j, res in enumerate(self.minvol.columns):
            v0 = V[res].to_numpy(dtype=np.float32)
            f[f"minvol_{res}"] = MV[:, :, j].ravel()
            f[f"volgap_{res}"] = rep(v0) - MV[:, :, j].ravel()           # volume at case start minus minimum at hour t
            f[f"volgap_{res}_wkmin"] = rep(v0 - MV[:, :, j].max(1))      # worst-case slack during the week
        return pd.DataFrame(f)

def add_constraint_features(f, starts, constraint_features):
    extra = constraint_features(starts)
    assert len(extra) == len(f)
    return pd.concat([f.reset_index(drop=True), extra], axis=1)

# ---- formerly src/features_context.py ----
# Temporal-context and cascade features on top of the margin features (see xgboost_context.ipynb).
#
# Context (per plant, from margin_end_<plant> = price - end-of-week water value of the upstream reservoir):
#   centred rolling mean / max / min over 3, 6 and 12 hours, number of positive-margin hours in the same day,
#   rank of the margin within the day, and run lengths: consecutive positive (negative) margin hours up to t and from t on.
# Cascade (per plant with a downstream reservoir): the downstream reservoir's water value at case start and end,
#   the upstream-minus-downstream difference, and price minus that difference ("cascade margin").

H = 168
# plant -> reservoir the plant discharges into (plant -> reservoir connections in Tokke_Vinje_topology.yaml)
DOWNSTREAM = {"Tokke": "Bandak", "Vinje": "Vinjevatn", "Haukeli": "Vinjevatn", "Byrte": "Byrtevatn",
              "Kjela": "Hyljelihyl", "Vesle Kjela": "Foersvatn", "Songa": "Totak"}   # Lio -> river, Hogga -> outlet
WINDOWS = (3, 6, 12)

def _run_lengths(pos):
    """pos: bool (n, H). Returns consecutive-True counts ending at t and starting at t."""
    n = pos.shape[0]
    back = np.zeros((n, H), dtype=np.float32); ahead = np.zeros((n, H), dtype=np.float32)
    for t in range(H):
        back[:, t] = np.where(pos[:, t], (back[:, t - 1] if t else 0) + 1, 0)
    for t in range(H - 1, -1, -1):
        ahead[:, t] = np.where(pos[:, t], (ahead[:, t + 1] if t < H - 1 else 0) + 1, 0)
    return back, ahead

def add_context_features(f, col="margin_end", prefix="ctx", zero=0.0, windows=WINDOWS):
    """Context of f"{col}_{plant}" (default: the end-of-week margin). `zero` is the ON/OFF threshold of that signal
    (0 for a margin, 1 for a price/water-value ratio)."""
    n = len(f) // H
    out = {}
    for plant in UPSTREAM:
        M = f[f"{col}_{plant}"].to_numpy(dtype=np.float32).reshape(n, H)
        for w in windows:
            out[f"{prefix}_mean{w}_{plant}"] = uniform_filter1d(M, size=w, axis=1, mode="nearest").ravel()
            out[f"{prefix}_max{w}_{plant}"] = maximum_filter1d(M, size=w, axis=1, mode="nearest").ravel()
            out[f"{prefix}_min{w}_{plant}"] = minimum_filter1d(M, size=w, axis=1, mode="nearest").ravel()
        day = M.reshape(n, 7, 24)
        out[f"{prefix}_poshours_day_{plant}"] = np.repeat((day > zero).sum(2), 24, axis=1).ravel()
        out[f"{prefix}_rank_day_{plant}"] = (day.argsort(2).argsort(2) / 23.0).reshape(n, H).ravel()
        pos_back, pos_ahead = _run_lengths(M > zero)
        neg_back, neg_ahead = _run_lengths(M <= zero)
        out[f"{prefix}_posrun_back_{plant}"] = pos_back.ravel(); out[f"{prefix}_posrun_ahead_{plant}"] = pos_ahead.ravel()
        out[f"{prefix}_negrun_back_{plant}"] = neg_back.ravel(); out[f"{prefix}_negrun_ahead_{plant}"] = neg_ahead.ravel()
    return pd.concat([f.reset_index(drop=True), pd.DataFrame(out)], axis=1)

def add_cascade_features(f):
    out = {}
    for plant, down in DOWNSTREAM.items():
        up = UPSTREAM[plant]
        out[f"casc_wv_down_{plant}"] = f[f"wv_{down}"].to_numpy()
        out[f"casc_wv_down_end_{plant}"] = f[f"wv_end_{down}"].to_numpy()
        diff_end = f[f"wv_end_{up}"].to_numpy() - f[f"wv_end_{down}"].to_numpy()
        out[f"casc_wvdiff_end_{plant}"] = diff_end
        out[f"casc_margin_{plant}"] = f["price"].to_numpy() - diff_end
    return pd.concat([f.reset_index(drop=True), pd.DataFrame(out)], axis=1)

# ---- formerly src/features_relative.py ----
# Scale-free versions of the margin signal (Feature_Engineering slides: pairwise transforms x./y, x-y).
#
# Why: prices and water values shift by an order of magnitude between years (2020 mean price 9 EUR, 2022 mean 211 EUR),
# so a margin of +2 EUR means "clearly profitable" in 2020 but "noise" in 2022. Trees split on absolute thresholds and
# transfer badly across regimes (rolling validation: 2020 report AUC 0.92 vs 0.98 in 2021/2022). Ratios and margins
# measured in units of the week's own price spread do not depend on the price level.

H = 168

def add_relative_features(f):
    n = len(f) // H
    price = f["price"].to_numpy(dtype=np.float32)
    spread = f["price_week_std"].to_numpy(dtype=np.float32) + 1.0
    out = {}
    for plant, res in UPSTREAM.items():
        wv0 = np.maximum(f[f"wv_{res}"].to_numpy(dtype=np.float32), 1.0)         # floor at 1 EUR: wv ~0 in 2020
        wv1 = np.maximum(f[f"wv_end_{res}"].to_numpy(dtype=np.float32), 1.0)
        margin_end = f[f"margin_end_{plant}"].to_numpy(dtype=np.float32)
        out[f"ratio_start_{plant}"] = price / wv0                                   # price / water value
        out[f"ratio_end_{plant}"] = price / wv1
        out[f"ratio_wk_{plant}"] = f["price_week_mean"].to_numpy(dtype=np.float32) / wv1
        out[f"margin_end_rel_{plant}"] = margin_end / spread                        # margin in units of weekly price spread
        out[f"poshours_wk_{plant}"] = np.repeat((margin_end.reshape(n, H) > 0).mean(1), H)   # share of profitable hours
    return pd.concat([f.reset_index(drop=True), pd.DataFrame(out)], axis=1)

# Absolute price / water-value levels. Dropping them forces the model onto scale-free features only.
LEVEL_PREFIXES = ("price", "wv_", "margin_")

def level_columns(f):
    keep = {"price_rank", "price_week_std"}   # already scale-free (rank) or needed as a spread measure
    return [c for c in f.columns if c.startswith(LEVEL_PREFIXES) and c not in keep and "_rel_" not in c]

# ---- formerly src/features_reservoir.py ----
# Reservoir-state features: fill fraction, weekly inflow volume, projected fill, water-value trend.
#
# Why: the error analysis showed that half of the hours have a clearly negative margin, yet 7% of them are ON
# (forced running: full reservoir, spill, min flow). Fill fraction and inflow-vs-free-capacity describe that
# pressure directly. max_vol per reservoir is read from data/extended/Tokke_Vinje_topology.yaml (reservoir section).

H = 168
# Mm3, "max_vol" of each reservoir in Tokke_Vinje_topology.yaml
MAX_VOL = {"Bandak": 174.2, "Vinjevatn": 9.36, "Byrtevatn": 76.71, "Botnedalsvatn": 54.17, "Kjelavatn": 84.5,
           "Staavatn": 48.0, "Venemo": 23.0, "Langesae": 78.0, "Hyljelihyl": 0.26, "Bordalsvatn": 184.2,
           "Foersvatn": 122.3, "Songavatn": 640.0, "Bitdalsvatn": 110.0, "Totak": 257.8, "Vaamarvatn": 26.2,
           "Langeidvatn": 32.1, "Vatjern": 0.3}

def add_reservoir_features(f):
    n = len(f) // H
    out = {}
    for res, vmax in MAX_VOL.items():
        vol = f[f"vol_{res}"].to_numpy(dtype=np.float32)
        inflow = f[f"inflow_{res}"].to_numpy(dtype=np.float32).reshape(n, H)
        wk_mm3 = np.repeat(inflow.sum(1) * 3600 / 1e6, H)                  # m3/s summed over the week -> Mm3
        out[f"fill_{res}"] = vol / vmax                                       # fraction of max volume at case start
        out[f"inflow_wk_{res}"] = wk_mm3
        out[f"fill_proj_{res}"] = (vol + wk_mm3) / vmax                       # fill at week end if nothing is produced
        wv0 = f[f"wv_{res}"].to_numpy(dtype=np.float32)
        wv1 = f[f"wv_end_{res}"].to_numpy(dtype=np.float32)
        out[f"wv_trend_{res}"] = wv1 - wv0                                    # optimiser's view of scarcity over the week
        out[f"wv_trend_rel_{res}"] = (wv1 - wv0) / (np.abs(wv0) + 1.0)
    for plant, res in UPSTREAM.items():                                      # 24 h window mean of the inflow feeding each plant
        inflow = f[f"inflow_{res}"].to_numpy(dtype=np.float32).reshape(n, H)
        out[f"inflow_d24_{plant}"] = np.repeat(inflow.reshape(n, 7, 24).mean(2), 24, axis=1).ravel()
    return pd.concat([f.reset_index(drop=True), pd.DataFrame(out)], axis=1)

# ---- formerly src/features_horizon.py ----
# Horizon-position features: expanding-window statistics of the scale-free margin over the remaining hours (t..167)
# and the elapsed hours (0..t) of the case window.
#
# Why: the same calendar hour gets a different decision in 36% of the overlapping cases that contain it, so the decision
# depends on what the optimiser sees ahead of hour t inside its one-week horizon, not only on the hour itself.
# Feature_Engineering slides: local/window features at several scales (here: the two one-sided windows that end the horizon).

H = 168

def add_horizon_features(f, col="margin_end_rel"):
    n = len(f) // H
    out = {}
    t = np.arange(1, H + 1, dtype=np.float32)                      # hours elapsed including t
    for plant in UPSTREAM:
        M = f[f"{col}_{plant}"].to_numpy(dtype=np.float32).reshape(n, H)
        pos = (M > 0).astype(np.float32)
        cs, cp = np.cumsum(M, 1), np.cumsum(pos, 1)                 # inclusive sums up to t
        tot, totp = cs[:, -1:], cp[:, -1:]
        out[f"hz_mean_behind_{plant}"] = (cs / t).ravel()
        out[f"hz_mean_ahead_{plant}"] = ((tot - cs + M) / (H - t + 1)).ravel()
        out[f"hz_pos_behind_{plant}"] = (cp / t).ravel()            # share of profitable hours so far
        out[f"hz_pos_ahead_{plant}"] = ((totp - cp + pos) / (H - t + 1)).ravel()
        out[f"hz_max_ahead_{plant}"] = np.maximum.accumulate(M[:, ::-1], axis=1)[:, ::-1].ravel()
        out[f"hz_min_ahead_{plant}"] = np.minimum.accumulate(M[:, ::-1], axis=1)[:, ::-1].ravel()
        # rank of hour t among the remaining hours (0 = cheapest remaining, 1 = most profitable remaining)
        ahead_rank = np.zeros_like(M)
        for tt in range(H):
            rem = M[:, tt:]
            ahead_rank[:, tt] = (rem < M[:, tt:tt + 1]).sum(1) / max(H - tt - 1, 1)
        out[f"hz_rank_ahead_{plant}"] = ahead_rank.ravel()
    return pd.concat([f.reset_index(drop=True), pd.DataFrame(out)], axis=1)

# ---- formerly src/features_water.py ----
# Water-budget interaction features: does hour t rank among the K most profitable hours of the week, where K is the
# number of hours the plant can run on the water it has (start volume + weekly inflow of the upstream reservoir, divided by
# the plant's full-load discharge from the turbine curves in Tokke_Vinje_topology.yaml)?
#
# Feature_Engineering slides, pairwise transforms (x - y): the rank of the hour minus the hours of water available.

H = 168
# m3/s at full load for the whole plant (sum over units of max discharge on the turbine efficiency curves)
Q_MAX = {"Hogga": 168.6, "Tokke": 128.0, "Lio": 14.0, "Byrte": 8.0, "Vinje": 165.0, "Haukeli": 2.1,
         "Songa": 52.0, "Kjela": 40.0, "Vesle Kjela": 20.0}

def add_water_features(f):
    n = len(f) // H
    rank_from_top = (1.0 - f["price_rank"].to_numpy(dtype=np.float32)) * (H - 1)     # 0 = most expensive hour of the week
    out = {}
    for plant, res in UPSTREAM.items():
        water = (f[f"vol_{res}"] + f[f"inflow_wk_{res}"]).to_numpy(dtype=np.float32) if f"vol_{res}" in f else \
                (f[f"fill_{res}"].to_numpy(dtype=np.float32) * MAX_VOL[res] + f[f"inflow_wk_{res}"].to_numpy(dtype=np.float32))
        k_hours = np.clip(water * 1e6 / (Q_MAX[plant] * 3600.0), 0, 2 * H)                # full-load hours of water
        spill_hours = np.clip((water - MAX_VOL[res]) * 1e6 / (Q_MAX[plant] * 3600.0), 0, 2 * H)   # hours needed to avoid spill
        pos_hours = f[f"poshours_wk_{plant}"].to_numpy(dtype=np.float32) * H
        out[f"water_k_hours_{plant}"] = k_hours
        out[f"water_rank_slack_{plant}"] = rank_from_top - k_hours          # <= 0: hour is within the water budget
        out[f"water_spill_slack_{plant}"] = rank_from_top - spill_hours     # <= 0: must run anyway to avoid spill
        out[f"water_vs_profit_{plant}"] = pos_hours - k_hours               # more profitable hours than water -> selective
    return pd.concat([f.reset_index(drop=True), pd.DataFrame(out)], axis=1)

def add_inflow_horizon_features(f):
    """Cumulative inflow to each plant's upstream reservoir from the case start to hour t and from t to the end of the
    week, in hours of full-load discharge. For reservoirs whose volume is a fake constant (Vatjern/Haukeli, Hyljelihyl)
    this is the only description of the reservoir state inside the week."""
    n = len(f) // H
    out = {}
    for plant, res in UPSTREAM.items():
        I = f[f"inflow_{res}"].to_numpy(dtype=np.float32).reshape(n, H) * 3600.0 / (Q_MAX[plant] * 3600.0)   # hours of discharge per hour of inflow
        cs = np.cumsum(I, 1)
        out[f"inflow_h_behind_{plant}"] = cs.ravel()
        out[f"inflow_h_ahead_{plant}"] = (cs[:, -1:] - cs + I).ravel()
        out[f"inflow_h_rel_{plant}"] = (I / (I.mean(1, keepdims=True) + 1e-3)).ravel()   # inflow now vs the week's mean
    return pd.concat([f.reset_index(drop=True), pd.DataFrame(out)], axis=1)


In [ ]:
# run from the repo root so that data/, results/ and submissions/ resolve (notebooks live in notebooks/)
import os, re, json, time
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from kaggle_metric import score

DEVICE = "cuda" if os.environ.get("XGB_DEVICE", "cpu") == "cuda" else "cpu"   # set XGB_DEVICE=cuda on a GPU machine
PARAMS = dict(learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, min_child_weight=5)
print("xgboost", xgb.__version__, "| device", DEVICE, "| seed", SEED)

## 1. Data and features

One function builds the table for the training and the test case start times.

In [ ]:
uc, price, inflow, volume, water_value = load("data/kernel")
minflow, minvol = load_constraints("data/extended")
test_map = pd.read_csv("prediction_mapping.csv", parse_dates=["starttime"])
sample = pd.read_csv("sample_submission.csv", nrows=1)
label_cols = [c for c in sample.columns if c != "Run No"]
units = list(dict.fromkeys(re.match(r"result_committed_(.+)_t\d+$", c).group(1) for c in label_cols))
N_UNITS = len(units)
Y = uc[label_cols].to_numpy(dtype=np.int8).reshape(len(uc), N_UNITS, H).transpose(0, 2, 1).reshape(-1, N_UNITS)
starts = uc["starttime"]
build_base = FeatureBuilder(price, inflow, volume, water_value)
constraints = ConstraintFeatures(minflow, minvol, volume)

In [ ]:
# ---- feature variants (formerly src/feature_experiments.py) ----
def add_system_features(f):
    """Cross-plant view: how many plants are profitable at hour t, and each plant relative to the system mean."""
    rel = np.stack([f[f"margin_end_rel_{p}"].to_numpy(dtype=np.float32) for p in UPSTREAM], 1)
    ratio = np.stack([f[f"ratio_end_{p}"].to_numpy(dtype=np.float32) for p in UPSTREAM], 1)
    out = {"sys_n_profitable": (ratio > 1).sum(1), "sys_mean_rel": rel.mean(1), "sys_max_rel": rel.max(1)}
    for j, p in enumerate(UPSTREAM):
        out[f"sys_diff_{p}"] = rel[:, j] - out["sys_mean_rel"]
    return pd.concat([f.reset_index(drop=True), pd.DataFrame(out)], axis=1)


def build_variants(base):
    """base = margin + constraint + context features (the model of notebook 06, 268 columns). Each variant is a function
    so that only the requested feature tables are materialised."""
    rel = add_relative_features(base)
    res = add_reservoir_features(base)
    both = add_reservoir_features(rel)
    ctx_cols = [c for c in both.columns if c.startswith("ctx_")]
    horizon = lambda: add_horizon_features(add_context_features(both.drop(columns=level_columns(both) + ctx_cols), col="margin_end_rel"))
    return {
        "context":         lambda: base,                                                  # reference (notebook 06)
        "relative":        lambda: rel,                                                   # + ratios, margin/spread, share of profitable hours
        "reservoir":       lambda: res,                                                   # + fill, weekly inflow, projected fill, wv trend
        "both":            lambda: both,
        "both_nolevel":    lambda: both.drop(columns=level_columns(both)),                # scale-free only: drop price / wv / margin levels
        "both_ratioctx":   lambda: add_context_features(both, col="ratio_end", prefix="rctx", zero=1.0),
        "rel_nolevel":     lambda: rel.drop(columns=level_columns(rel)),
        "relctx_nolevel":  lambda: add_context_features(both.drop(columns=level_columns(both) + ctx_cols), col="margin_end_rel"),  # notebook 08
        "horizon_nolevel": lambda: horizon(),                                             # + expanding-window stats ahead / behind within the case
        "hz_ctx24":        lambda: add_context_features(horizon(), col="margin_end_rel", prefix="ctxl", windows=(24, 48)),
        "hz_nocal":        lambda: horizon().drop(columns=["doy", "month"]),
        "hz_system":       lambda: add_system_features(horizon()),
        "hz_water":        lambda: add_water_features(horizon()),
        "hz_inflow":       lambda: add_inflow_horizon_features(horizon()),
    }


def base_features(start_times):
    """Margin + constraint + context features; the starting point of every variant."""
    f = add_margin_features(build_base(start_times))
    f = add_constraint_features(f, start_times, constraints)
    return add_context_features(f)

VARIANT = "horizon_nolevel"
FINAL_PARAMS = {**PARAMS, "min_child_weight": 50, "reg_lambda": 20, "colsample_bytree": 0.5}
N_SEEDS = 5

def build_features(start_times):
    return build_variants(base_features(start_times))[VARIANT]()

t0 = time.time()
X = build_features(starts)
X_test = build_features(test_map["starttime"])
assert list(X.columns) == list(X_test.columns) and len(X) == len(Y)
assert not X.isna().any().any() and not X_test.isna().any().any()
print(f"X {X.shape}, X_test {X_test.shape}, built in {time.time() - t0:.0f}s")

## 2. Honest validation: rounds on 2021, report on 2022

In [ ]:
t0 = time.time()
res = honest_fit(X, Y, starts, FINAL_PARAMS, tune_year=2021, report_year=2022, device=DEVICE)
print(f"rounds chosen on 2021: {res['n_rounds']}   2021 AUC {res['tune_auc']:.4f}   2022 report micro AUC {res['report_auc']:.4f}   [{time.time() - t0:.0f}s]")
per_unit = pd.Series(res["per_unit"], index=units, name="AUC 2022").sort_values()
per_unit.round(4).to_frame().T

In [ ]:
imp = pd.Series(res["model"].get_booster().get_score(importance_type="gain"), name="gain").sort_values(ascending=False)
imp.head(15).round(0).to_frame().T

## 3. Bagged final fit on all cases and the submission file

In [ ]:
t0 = time.time()
preds = []
for seed in range(N_SEEDS):
    m = xgb.XGBClassifier(n_estimators=res["n_rounds"], tree_method="hist", device=DEVICE, random_state=SEED + seed, **FINAL_PARAMS).fit(X, Y)
    preds.append(m.predict_proba(X_test)); print(f"seed {seed} done [{time.time() - t0:.0f}s]", flush=True)
pred = np.mean(preds, axis=0)
wide = pred.reshape(len(test_map), H, N_UNITS).transpose(0, 2, 1).reshape(len(test_map), -1)
submission = pd.DataFrame(wide, columns=label_cols); submission.insert(0, "Run No", test_map["Run No"].to_numpy())
assert list(submission.columns) == list(sample.columns)
assert (submission["Run No"].to_numpy() == test_map["Run No"].to_numpy()).all()
assert ((wide >= 0) & (wide <= 1)).all() and not np.isnan(wide).any()

os.makedirs("submissions", exist_ok=True); os.makedirs("results", exist_ok=True)
fname = f"submissions/{pd.Timestamp.today():%Y-%m-%d}_xgb_horizon_regd_bag{N_SEEDS}_cv{res['report_auc']:.4f}.csv"
submission.to_csv(fname, index=False)
json.dump({"variant": VARIANT, "params": FINAL_PARAMS, "seeds": N_SEEDS, "report_2022_micro_auc": res["report_auc"], "tune_2021_auc": res["tune_auc"],
           "n_rounds": res["n_rounds"], "n_features": int(X.shape[1]), "per_unit_auc": dict(zip(units, map(float, res["per_unit"])))},
          open("results/metrics_final.json", "w"), indent=2)
print(f"wrote {fname}  shape {submission.shape}  mean prediction {wide.mean():.3f}")
submission.iloc[:3, :6]

## 4. Summary

| | |
|---|---|
| model | XGBoost, one booster per unit, rows = (case, hour), 384 scale-free features |
| validation | rounds on 2021, report on 2022; rolling validation 2020–2022 for every decision (notebook 09) |
| final fit | 5 seeds on all 2015–2022 cases, averaged |
| Kaggle public | the unbagged `relctx_nolevel` model scored 0.99106; the two bagged horizon submissions correlate 0.996 with it |